# Exercise on the value of unsupervised constructed features for training a classifier with few labeled examples


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deep-learning-ids/deep_learning_fs26/blob/main/notebooks/04_classification_transfer_learning_few_labels_keras_torch.ipynb)

To get unsupervised constructed features of an image, we can use a pretrained CNN as feature extractor.

We have done this to extract features from 100 Cifar10 images.  As pretrained CNN we use a VGG16 architecture that was trained on ImageNet data and was the second winner of the ImageNet competition in 2014.

As a check on the quality of the feature representation of the CIFAR10 data, we will use once the pixel-features and once the VGG-features to train a classifier using this 100 labeled data (on average 10 per class). If the VGG-feature are indeed better than the raw pixel values, we would expect to achieve a better classifier when using the VGG-feature compared to the pixel feature.

a) Which accuracy would you expect for a classifier which cannot distinguish between the 10 classes and is only guessing?



b) Go through the code in **SECTION 1)** which is used to set-up, train, and evaluate a CNN classifier using the raw pixel features . Discuss your thoughts on the achieved accuracy (e.g. with your neighbor).


c) in **SECTION 2)** Now we use the unsupervised constructed VGG features. We want to check, if these VGG features are good enough to train a classifier with only few labeled data and still get a satisfying performance. For this purpose, please complete the code to set up a fully connected NN and run the provided subsequent code to train it and determine its accuracy on the test set. Compare it to the accuracy which we achieve with a RF. Discuss the results (e.g. with your neighbor).





### 🔑 **Solution:**

<details>
  <summary>🔑 a) </summary>


**Solution: 10%**

</details>

<details>
  <summary>🔑 b) </summary>


**Solution: The accuracy is with around 20% better then guessing but still very bad. However, this is not surprising since the resolution of the images are very low and it is alread by eye quite difficult to distinguish between the classes. Moreover, we have  only very few training examples (only 10 per class), quite bad features (the raw pixel values) and a model with many parameters (around 45k parameter).**

</details>

<details>
  <summary>🔑 c) </summary>


**Solution: For code completion see below. The accuracy of the fcNN on the VGG features is with more than 55% much better than the accuray of the from scratch trained CNN which was 20%. This implies that the VGG-features are quite good and more informative than the raw pixel features. With the RF we achieve a similar performance.**

</details>

# Section 1

## Imports

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.image as imgplot
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pylab import *

import time
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch # not needed yet

print(f'Keras_version: {keras.__version__}')# 3.5.0
print(f'torch_version: {torch.__version__}')# 2.5.1+cu121
print(f'keras backend: {keras.backend.backend()}')

# Keras Building blocks
from keras.models import Sequential
from keras.layers import Dense, Convolution2D, MaxPooling2D, Flatten , Activation
from keras.optimizers import SGD
from keras.utils import to_categorical
from keras import optimizers

import sys

## CIFAR Data preparation

In [ ]:
"""
Load the CIFAR-10 dataset for training purposes.

CIFAR-10 is a standard image classification dataset consisting of:
- 60,000 32x32 color images
- 10 different classes
- 50,000 training images
- 10,000 test images

This script:
1. Imports the CIFAR-10 dataset loader from Keras.
2. Downloads (if not already cached) and loads the dataset.
3. Separates the dataset into training and testing sets.
4. Deletes the test set to free memory since it is not needed.
"""

# Import CIFAR-10 dataset loader from Keras
from keras.datasets import cifar10

# Load dataset
# x_train: Training images (shape: 50000, 32, 32, 3)
# y_train: Training labels (shape: 50000, 1)
# x_test: Test images (shape: 10000, 32, 32, 3)
# y_test: Test labels (shape: 10000, 1)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Delete test data to conserve memory if only training data is needed
del [x_test, y_test]

In [ ]:
"""
Create a balanced subset of the training dataset by:

1. Iterating over each unique class label in y_train.
2. Randomly selecting 100 images per class (without replacement).
3. Storing the selected indices.
4. Subsetting x_train and y_train using those indices.

This results in:
- 100 samples per class
- Total samples = 100 × number_of_classes (e.g., 1000 for CIFAR-10)

A fixed random seed is used for reproducibility.
"""

# Set random seed for reproducibility
np.random.seed(seed=222)

# Initialize empty array to store selected indices
# NOTE: int8 is NOT ideal here for CIFAR-10 because indices can exceed 127.
# It is safer to use int32 or int64.
idx = np.empty(0, dtype="int32")

# Loop over each unique class label
for i in range(0, len(np.unique(y_train))):

    # Find indices where label equals current class i
    class_indices = np.where(y_train[0:len(y_train)] == i)[0]

    # Randomly select 100 samples from this class (without replacement)
    selected_indices = np.random.choice(class_indices, 100, replace=False)

    # Append selected indices to idx array
    idx = np.append(idx, selected_indices)

# Subset training data to only selected samples
x_train = x_train[idx]
y_train = y_train[idx]

In [ ]:
"""
Inspect the shape and class distribution of the balanced training subset.

This code:
1. Prints the shape of x_train (image data).
2. Prints the shape of y_train (labels).
3. Prints the unique class labels and the number of samples per class.

This verifies that:
- The dataset size matches expectations.
- Each class contains exactly 100 samples.
"""

# Print the shape of training images
print(x_train.shape)
# Expected (for CIFAR-10 subset): (1000, 32, 32, 3)
# 100 samples × 10 classes = 1000 total images

# Print the shape of training labels
print(y_train.shape)
# Expected: (1000, 1)

# Print unique labels and their counts
print(np.unique(y_train, return_counts=True))
# Expected output:
# (array([0,1,2,3,4,5,6,7,8,9]), array([100,100,100,100,100,100,100,100,100,100]))

In [ ]:
"""
Create a smaller training subset by:

1. Iterating over each unique class label in y_train.
2. Randomly selecting 10 samples per class (without replacement).
3. Storing the selected indices.
4. Creating new training subsets (x_train_new, y_train_new).

This results in:
- 10 samples per class
- Total samples = 10 × number_of_classes (e.g., 100 for CIFAR-10)

A fixed random seed ensures reproducibility.
"""


# Set random seed for reproducibility
np.random.seed(seed=123)

# IMPORTANT: Use int32 or int64 (NOT int8 — it will overflow for large indices)
idx_train = np.empty(0, dtype="int32")

# Loop over each class label
for i in range(0, len(np.unique(y_train))):

    # Get indices of samples belonging to class i
    class_indices = np.where(y_train == i)[0]

    # Randomly select 10 samples from this class
    selected_indices = np.random.choice(class_indices, 10, replace=False)

    # Append to index list
    idx_train = np.append(idx_train, selected_indices)

# Create new training subset
x_train_new = x_train[idx_train]
y_train_new = y_train[idx_train]

In [ ]:
"""
Create a complementary test set by removing the selected training indices.

This code:
1. Removes the samples indexed by idx_train from x_train and y_train.
2. Stores the remaining samples as x_test_new and y_test_new.

Effectively:
- x_train_new / y_train_new contain the selected subset (e.g., 10 per class).
- x_test_new / y_test_new contain the remaining samples.
"""

# Remove selected training samples to form the test set
x_test_new = np.delete(x_train, idx_train, axis=0)
y_test_new = np.delete(y_train, idx_train, axis=0)

In [ ]:
"""
Create a validation set by:

1. Iterating over each unique class label in y_test_new.
2. Randomly selecting 10 samples per class (without replacement).
3. Storing the selected indices.
4. Creating validation subsets (x_valid_new, y_valid_new).

This results in:
- 10 samples per class in validation
- Total validation samples = 10 × number_of_classes
- Reproducible split using a fixed random seed
"""

import numpy as np

# Set seed for reproducibility
np.random.seed(seed=127)

# IMPORTANT: Do NOT use int8 (will overflow). Use int32 or default Python list.
idx_valid = []

# Loop over each class label
for i in np.unique(y_test_new):

    # Get indices of current class
    class_indices = np.where(y_test_new == i)[0]

    # Randomly select 10 samples
    selected = np.random.choice(class_indices, 10, replace=False)

    idx_valid.extend(selected)

# Convert to numpy array
idx_valid = np.array(idx_valid)

# Create validation set
x_valid_new = x_test_new[idx_valid]
y_valid_new = y_test_new[idx_valid]

In [ ]:
"""
Remove validation samples from the temporary test set to create
the final test dataset.

This code:
1. Deletes the indices used for validation (idx_valid)
   from x_test_new and y_test_new.
2. The remaining samples become the final test set.

After this step:
- x_train_new  → Training set
- x_valid_new  → Validation set
- x_test_new   → Final test set
"""

# Remove validation samples from test set
x_test_new = np.delete(x_test_new, idx_valid, axis=0)
y_test_new = np.delete(y_test_new, idx_valid, axis=0)

In [ ]:
"""
Reshape the dataset splits to ensure they match the expected
4D tensor format for CNN input:

Format: (num_samples, height, width, channels)

For CIFAR-10:
- Image size = 32x32
- Channels = 3 (RGB)

Expected dataset sizes:
- Training set:   100 samples  (10 per class × 10 classes)
- Validation set: 100 samples  (10 per class × 10 classes)
- Test set:       800 samples  (80 per class × 10 classes)
"""

# Reshape training set
x_train_new = np.reshape(x_train_new, (100, 32, 32, 3))

# Reshape validation set
x_valid_new = np.reshape(x_valid_new, (100, 32, 32, 3))

# Reshape test set
x_test_new = np.reshape(x_test_new, (800, 32, 32, 3))

In [ ]:
"""
Verify class balance in each dataset split.

This code prints:
1. Unique class labels present in each split.
2. The number of samples per class.

It confirms that the stratified sampling worked correctly.
"""

# Check training set class distribution
print(np.unique(y_train_new, return_counts=True))

# Check validation set class distribution
print(np.unique(y_valid_new, return_counts=True))

# Check test set class distribution
print(np.unique(y_test_new, return_counts=True))

In [ ]:
"""
Convert integer class labels into one-hot encoded vectors.

`to_categorical()` transforms class labels from shape:
    (num_samples, 1)
or
    (num_samples,)
into one-hot encoded format:
    (num_samples, num_classes)

For CIFAR-10:
- num_classes = 10
- Each label becomes a 10-dimensional vector
- The correct class index is set to 1, others to 0

Example:
Class label: 3
One-hot:      [0,0,0,1,0,0,0,0,0,0]
"""

from keras.utils import to_categorical

# Convert labels to one-hot encoding
y_train_new = to_categorical(y_train_new, 10)
y_valid_new = to_categorical(y_valid_new, 10)
y_test_new  = to_categorical(y_test_new, 10)



In [ ]:
"""
Verify final dataset shapes after:
- Stratified splitting
- Reshaping image tensors
- One-hot encoding labels

This confirms:
1. Image tensors are 4D → (samples, 32, 32, 3)
2. Labels are one-hot encoded → (samples, 10)
"""

print(x_train_new.shape)
print(y_train_new.shape)

print(x_valid_new.shape)
print(y_valid_new.shape)

print(x_test_new.shape)
print(y_test_new.shape)

In [ ]:
"""
Center and standardize image data using training set statistics.

Standardization formula:
    X_normalized = (X - mean) / (std + epsilon)

Where:
- mean  → computed from training data only
- std   → computed from training data only
- epsilon (0.0001) → prevents division by zero

Important:
We compute mean and std ONLY from the training set to avoid
data leakage into validation and test sets.
"""

# Compute per-pixel mean and standard deviation from training data
X_mean = np.mean(x_train_new, axis=0)
X_std  = np.std(x_train_new, axis=0)

# Normalize training data
x_train_new = (x_train_new - X_mean) / (X_std + 0.0001)

# Normalize validation data using training statistics
x_valid_new = (x_valid_new - X_mean) / (X_std + 0.0001)

# Normalize test data using training statistics
x_test_new = (x_test_new - X_mean) / (X_std + 0.0001)

## Baseline 1: use raw images to train a Random Forest model

In [ ]:
"""
Prepare image data for a Random Forest classifier by flattening each image.

Why this is needed:
- Random Forests (and most classical ML models) expect a 2D feature matrix:
    shape = (num_samples, num_features)
- CIFAR-10 images are 4D tensors:
    shape = (num_samples, 32, 32, 3)
- Flattening converts each 32x32 RGB image into a single feature vector of length:
    32 * 32 * 3 = 3072

Resulting shapes:
- x_train_rf: (N_train, 3072)
- x_valid_rf: (N_valid, 3072)
- x_test_rf:  (N_test, 3072)
"""

# Flatten images into feature vectors for Random Forest input
x_train_rf = x_train_new.reshape(len(x_train_new), 32 * 32 * 3)
x_valid_rf = x_valid_new.reshape(len(x_valid_new), 32 * 32 * 3)
x_test_rf  = x_test_new.reshape(len(x_test_new),  32 * 32 * 3)

In [ ]:
"""
Train a Random Forest classifier on the flattened CIFAR-10 images.

Steps:
1. Import RandomForestClassifier from sklearn.
2. Initialize the classifier with default hyperparameters.
3. Convert one-hot encoded labels back to integer class labels.
4. Fit the model using training data.

Note:
- RandomForest expects labels as a 1D array of class indices,
  not one-hot encoded vectors.
- np.argmax(..., axis=1) converts one-hot labels back to integers.
"""

from sklearn.ensemble import RandomForestClassifier


# Initialize Random Forest with default parameters
clf = RandomForestClassifier()

# Train model
clf.fit(x_train_rf, np.argmax(y_train_new, axis=1))

In [ ]:
"""
Evaluate the trained Random Forest model.

Steps:
1. Predict class labels on the test set.
2. Compute the confusion matrix.
3. Compute classification accuracy.
4. Display the confusion matrix.

Note:
- y_test_new is one-hot encoded.
- np.argmax(..., axis=1) converts it back to integer labels.
"""

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


# Predict on test set
pred = clf.predict(x_test_rf)

# Convert one-hot test labels back to class indices
y_test_labels = np.argmax(y_test_new, axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test_labels, pred)

# Compute accuracy manually
acc_fc = np.sum(pred == y_test_labels) / len(y_test_labels)
print("Accuracy = ", acc_fc)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='viridis')
plt.title('Confusion Matrix, Baseline 1, RF')
plt.show()

## Setting up the the CNN classifier based on raw image data

In [ ]:
"""
Import Keras components needed to build a CNN (Convolutional Neural Network)
for image classification using raw image tensors.

Key components:
- Sequential: A simple linear stack of layers (common for basic CNNs).
- Dense: Fully-connected (classification) layers, usually near the end.
- Activation: Applies non-linearities (e.g., ReLU, softmax).
- Dropout: Regularization technique to reduce overfitting by randomly
  dropping units during training.
- BatchNormalization: Stabilizes and speeds up training by normalizing
  layer activations.
- Convolution2D (Conv2D): Learns spatial features using convolution filters.
- MaxPooling2D: Downsamples feature maps to reduce spatial size and compute.
- Flatten: Converts 3D feature maps into a 1D vector for Dense layers.

Note:
- In many modern Keras examples, Convolution2D is aliased as Conv2D.

"""

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout, BatchNormalization
from keras.layers import Convolution2D, MaxPooling2D, Flatten


In [ ]:
"""
Define hyperparameters and architectural settings for the CNN model.

These parameters control:
- Training behavior (batch size, epochs)
- Model output structure (number of classes)
- Image dimensions
- Convolution and pooling configuration
"""

# Training hyperparameters
batch_size = 10        # Number of samples processed before updating weights
nb_classes = 10        # CIFAR-10 has 10 output classes
nb_epoch = 30          # Number of full passes through the training dataset

# Image dimensions (CIFAR-10 images are 32x32 RGB)
img_rows, img_cols = 32, 32

# Convolution settings
kernel_size = (3, 3)   # 3x3 convolution filters (standard choice in CNNs)

# Input shape for the first convolutional layer
# Format: (height, width, channels)
input_shape = (img_rows, img_cols, 3)

# Pooling settings
pool_size = (2, 2)     # 2x2 max pooling reduces spatial dimensions by half

In [ ]:
"""
Define and compile a Convolutional Neural Network (CNN) for CIFAR-10 classification.

Architecture overview:
- Two convolution blocks with increasing filter counts (8 -> 16)
- Each conv block uses:
  Conv2D -> BatchNorm -> ReLU (repeated) -> MaxPool
- Flatten feature maps to a vector
- A small fully-connected (Dense) layer with dropout regularization
- Softmax output layer for 10-class classification

Notes:
- BatchNormalization helps stabilize and speed up training.
- Dropout helps reduce overfitting (important since the training set is small).
- Softmax + categorical_crossentropy is correct because labels are one-hot encoded.
"""

model = Sequential()

# --- Convolution Block 1 (8 filters) ---
# Input: (32, 32, 3)
model.add(Convolution2D(8, kernel_size, padding='same', input_shape=input_shape))
model.add(BatchNormalization())
model.add(Activation('relu'))

model.add(Convolution2D(8, kernel_size, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))

# Downsample spatial size: (32,32) -> (16,16)
model.add(MaxPooling2D(pool_size=pool_size))


# --- Convolution Block 2 (16 filters) ---
# Input: (16, 16, 8)
model.add(Convolution2D(16, kernel_size, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))

model.add(Convolution2D(16, kernel_size, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))

# Downsample spatial size: (16,16) -> (8,8)
model.add(MaxPooling2D(pool_size=pool_size))


# --- Classifier Head ---
# Flatten feature maps: (8, 8, 16) -> 1024 features
model.add(Flatten())

# Dense layer (small to reduce overfitting)
model.add(Dense(40))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(Activation('relu'))

# Output layer: 10 classes
model.add(Dense(nb_classes))
model.add(Activation('softmax'))

# Compile model:
# - categorical_crossentropy: correct for one-hot labels
# - adam: widely used adaptive optimizer
# - accuracy: report classification accuracy during training/evaluation
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
"""
Display a summary of the CNN architecture.

model.summary() prints:
- Layer names and types
- Output shape after each layer
- Number of trainable parameters per layer
- Total parameters in the model
- Trainable vs non-trainable parameters

This is useful for:
- Verifying tensor dimensions
- Checking parameter count
- Debugging architecture issues
"""

model.summary()

In [ ]:
"""
Train the CNN model.

model.fit() performs supervised training using:
- Training data (x_train_new, y_train_new)
- Validation data (x_valid_new, y_valid_new)

Arguments explained:
- batch_size=10:
    Number of samples per gradient update.
    With 100 training samples → 10 updates per epoch.

- epochs=30:
    Model will iterate over the full training dataset 30 times.

- verbose=2:
    Prints one line per epoch (clean output).

- validation_data:
    Used to evaluate model performance after each epoch.
    Helps monitor overfitting.

- shuffle=True:
    Shuffles training data at the start of each epoch
    (important for small datasets).
"""

history = model.fit(
    x_train_new, y_train_new,
    batch_size=10,
    epochs=30,
    verbose=2,
    validation_data=(x_valid_new, y_valid_new),
    shuffle=True
)

### Evaluation of the CNN classifier that was trained on raw image data

In [ ]:
"""
Evaluate the trained CNN on the test set.

Steps:
1. Generate predicted class probabilities.
2. Convert probabilities to predicted class indices.
3. Compute confusion matrix.
4. Compute test accuracy.
5. Display confusion matrix.

Since labels are one-hot encoded, we use np.argmax(...)
to convert both predictions and true labels back to class indices.
"""

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

# Predict class probabilities on test set
pred = model.predict(x_test_new)

# Convert one-hot labels and predictions to class indices
y_true = np.argmax(y_test_new, axis=1)
y_pred = np.argmax(pred, axis=1)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Accuracy
acc_cnn = np.sum(y_true == y_pred) / len(y_true)
print("Accuracy = ", acc_cnn)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='viridis')
plt.title('Confusion Matrix, Baseline 1, CNN')
plt.show()

# Section 2

## Getting the VGG features for CIFAR

In [ ]:
"""
Download precomputed CIFAR-10 embeddings (feature representations).

Instead of training from raw pixels, this approach uses
pretrained embeddings — typically extracted from a larger,
pretrained neural network.

Why embeddings?
- They contain higher-level semantic features.
- They often perform better than raw pixels with small datasets.
- They reduce dimensionality compared to flattened images.

This code:
1. Checks if the embedding file exists locally.
2. Downloads it if missing.
3. Lists file details to confirm download.
"""

# Downloading embeddings
import urllib
import os

# Check if file already exists
if not os.path.isfile('cifar_EMB_1000.npz'):
    urllib.request.urlretrieve(
        "https://www.dropbox.com/s/si287al91c1ls0d/cifar_EMB_1000.npz?dl=1",
        "cifar_EMB_1000.npz"
    )

# Display file information (Jupyter notebook command)
%ls -hl cifar_EMB_1000.npz

In [ ]:
"""
Load pretrained embedding features from the downloaded .npz file.

The .npz file is a compressed NumPy archive.
When saved without explicit names, arrays are stored as:
    "arr_0", "arr_1", ...

This code:
1. Loads the archive.
2. Extracts the first stored array (arr_0).
3. Stores it in vgg_features_cifar.

These features were likely extracted using a pretrained VGG-style CNN,
so instead of raw pixels (3072 features), we now have
high-level learned representations.
"""


# Load compressed NumPy archive
Data = np.load("cifar_EMB_1000.npz")

# Extract embedding features
vgg_features_cifar = Data["arr_0"]

In [ ]:
"""
Split pretrained embedding features into
train / validation / test sets using the same indices
previously defined for the image data.

Important:
- idx_train → indices of the 10 samples per class (training set)
- idx_valid → indices of the 10 samples per class (validation set)
- Remaining samples → final test set

This ensures:
- Exact alignment between raw images and embeddings
- Fair comparison across RF, CNN, and pretrained models
"""

# Training embeddings (10 per class → 100 samples total)
vgg_features_cifar_train = vgg_features_cifar[idx_train]

# Remove training samples to create temporary test pool
vgg_features_cifar_test = np.delete(vgg_features_cifar, idx_train, axis=0)

# Validation embeddings (10 per class → 100 samples)
vgg_features_cifar_valid = vgg_features_cifar_test[idx_valid]

# Remove validation samples → final test set (80 per class → 800 samples)
vgg_features_cifar_test = np.delete(vgg_features_cifar_test, idx_valid, axis=0)


In [ ]:
"""
Verify the shapes of the pretrained embedding splits.

This confirms:
- Train / validation / test sizes are correct
- Feature dimensionality is consistent
- Splitting aligned properly with earlier indices
"""

print(vgg_features_cifar_train.shape)
print(vgg_features_cifar_valid.shape)
print(vgg_features_cifar_test.shape)

## Baseline 2: use VGG feature to train a Random Forest model

In [ ]:
"""
Train a Random Forest classifier using pretrained VGG embeddings.

Instead of raw pixel values (3072 features),
we now train on high-level semantic feature vectors extracted
from a pretrained CNN.

Steps:
1. Initialize RandomForestClassifier.
2. Convert one-hot labels back to integer class labels.
3. Fit the model on embedding features.
"""


# Initialize Random Forest (default parameters)
clf = RandomForestClassifier()

# Train on pretrained embeddings
clf.fit(
    vgg_features_cifar_train,
    np.argmax(y_train_new, axis=1)   # convert one-hot → class index
)

In [ ]:
"""
Evaluate the Random Forest trained on pretrained VGG embeddings.

Steps:
1. Predict class labels on the test embedding set.
2. Convert one-hot encoded true labels back to class indices.
3. Compute confusion matrix.
4. Compute overall test accuracy.
5. Visualize the confusion matrix.

This allows direct comparison with:
- RF on raw pixels
- CNN trained from scratch
"""



# Predict using embedding-based RF
pred = clf.predict(vgg_features_cifar_test)

# Convert one-hot labels to integer labels
y_true = np.argmax(y_test_new, axis=1)

# Confusion matrix
cm = confusion_matrix(y_true, pred)

# Accuracy
acc_vgg_rf = np.sum(pred == y_true) / len(y_true)
print("Accuracy = ", acc_vgg_rf)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='viridis')
plt.title('Confusion Matrix, VGG Baseline 2, RF')
plt.show()


## Setting up the the NN classifier based on VGG feature

### 🔧 Your Task:
  - fill in the missing Code below


In [ ]:
"""
Define a Neural Network (MLP) classifier on top of pretrained VGG embeddings.

Here we are NOT training a CNN on pixels.
Instead, we treat each image as a fixed feature vector (embedding)
and train a standard feed-forward neural network (Dense layers).

Assumptions:
- Each embedding is 4096-dimensional (common for VGG-style FC features).
  This is why input_shape=(4096,).

Architecture so far:
- Dense(200) -> BatchNorm -> Dropout(0.5) -> ReLU
- Dense(200)  (a second hidden layer; typically you'll add BN/ReLU after it too)

Regularization:
- Dropout(0.5) is strong regularization (useful because training set is tiny).
- BatchNormalization helps stabilize training.

Note:
- This snippet defines only the "feature-to-hidden" part.
  You will still need:
  - activation (and often BN+Dropout) after the second Dense
  - final output layer Dense(nb_classes) + softmax
  - compile() with categorical_crossentropy
"""

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout, BatchNormalization

model = Sequential()

# First hidden layer: maps 4096-d embeddings -> 200 units
model.add(Dense(200, input_shape=(4096,)))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Activation('relu'))

# Second hidden layer (activation/regularization typically added after this too)
model.add(Dense(200))

# we still need to add the last layers to get the predictions on the 10 classes

#######🔧  your code here  ######



####### 🔧 end of your code ######


In [ ]:
# @title 🔑 Solution Code { display-mode: "form" }

"""
Add the output layer to the MLP classifier built on top of VGG embeddings.

This layer:
- Has `nb_classes` units (10 for CIFAR-10).
- Uses softmax activation to output class probabilities.

Because labels are one-hot encoded,
we will use `categorical_crossentropy` as the loss function.
"""

# Output layer: 10 classes
model.add(Dense(nb_classes))
model.add(Activation('softmax'))

In [ ]:
"""
Compile and summarize the MLP model built on top of VGG embeddings.

Compilation:
- loss='categorical_crossentropy'
    Correct because labels are one-hot encoded.
- optimizer='adam'
    Adaptive gradient optimizer (good default choice).
- metrics=['accuracy']
    Track classification accuracy during training.

model.summary():
- Displays layer types
- Output shapes
- Parameter counts
- Total trainable parameters
"""

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
"""
Train the MLP classifier on top of pretrained VGG embeddings.

Inputs:
- vgg_features_cifar_train → shape (100, D)
- y_train_new              → shape (100, 10)  (one-hot labels)

Validation:
- vgg_features_cifar_valid → shape (100, D)
- y_valid_new              → shape (100, 10)

Hyperparameters:
- batch_size=10  → 10 gradient updates per epoch (100 samples total)
- epochs=20      → 20 full passes over training data
- verbose=2      → one clean log line per epoch
- shuffle=True   → reshuffle training data each epoch (important for small datasets)
"""

history = model.fit(
    vgg_features_cifar_train,
    y_train_new,
    batch_size=10,
    epochs=20,
    verbose=2,
    validation_data=(vgg_features_cifar_valid, y_valid_new),
    shuffle=True
)

### Evaluation of the NN classifier that was trained on VGG features


### 🔧 Your Task:
  - fill in the missing Code below, from the predicitons plot the confusion matrix and calculate accuracy

  `cm =`
  
 ` acc_vgg_nn=`


In [ ]:
"""
Generate class probability predictions for the test embeddings
using the trained MLP model.
"""

pred = model.predict(vgg_features_cifar_test)

# get confusion matrix
#### we now want to get the confusion matrix for the predictions on the test data

#######🔧  your code here  ######


####### 🔧 end of your code ######

In [ ]:
# @title 🔑 Solution Code { display-mode: "form" }
"""
Evaluate the MLP (Neural Network) trained on VGG embeddings.

Steps:
1. Convert one-hot encoded true labels to class indices.
2. Convert predicted probabilities to predicted class indices.
3. Compute confusion matrix.
4. Compute overall accuracy.
5. Visualize confusion matrix.
"""

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

# Convert to class indices
y_true = np.argmax(y_test_new, axis=1)
y_pred = np.argmax(pred, axis=1)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Accuracy
acc_vgg_nn = np.sum(y_true == y_pred) / len(y_true)
print("Accuracy = ", acc_vgg_nn)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='viridis')
plt.title('Confusion Matrix, VGG Baseline 2, NN')
plt.show()

In [ ]:
"""
Print a clean comparison table of all model accuracies.

This summarizes performance across:
- Feature type (RAW pixels vs VGG embeddings)
- Model type (Random Forest, CNN, Neural Network)

Accuracies:
- acc_fc       → RF on raw pixels
- acc_cnn      → CNN trained from scratch
- acc_vgg_rf   → RF on VGG embeddings
- acc_vgg_nn   → NN on VGG embeddings
"""

print(f"""
Data         | Model           | Accuracy
-------------|----------------|----------
RAW          | Random Forest  | {acc_fc:.4f}
RAW          | CNN            | {acc_cnn:.4f}
VGG_features | Random Forest  | {acc_vgg_rf:.4f}
VGG_features | Neural Network | {acc_vgg_nn:.4f}
""")
